In [ ]:
!pip install transformers torch

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [ ]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
)
model.to(device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [ ]:
def hardcoded_response(user_input):
    user_input = user_input.lower().strip()
    if "what is your name" in user_input:
        return "🤖 Chatbot: I am a chatbot powered by TinyLlama."
    elif user_input in ["exit", "quit"]:
        return "🤖 Chatbot: Goodbye! Have a great day."
    elif user_input in ["hi", "hello", "hey"]:
        return "🤖 Chatbot: Hello! How can I help you today?"
    elif "thank" in user_input:
        return "🤖 Chatbot: You're welcome! If you have any other questions, feel free to ask."
    return None

In [ ]:
chat_history = ""
for step in range(5):
    user_input = input("User: ")
    hardcoded = hardcoded_response(user_input)
    if hardcoded:
        print(hardcoded)
        chat_history += f"<|user|>\n{user_input}\n<|chatbot|>\n{hardcoded}\n"
        continue
    prompt = f"<|user|>\n{user_input}\n<|chatbot|>\n"
    chat_history += prompt
    inputs = tokenizer(chat_history, return_tensors="pt").to(device)
    output = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=True,
        top_k=40,
        top_p=0.9,
        temperature=0.7,
        repetition_penalty=1.1
    )
    response = tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )
    print(f"🤖 Chatbot: {response}")
    chat_history += response + "\n"

User: Who invented Python?
🤖 Chatbot: Python is an interpreted, object-oriented, high-level programming language. It was invented by Guido van Rossum in 1989 at the University of Cambridge and first released as a free software project in 1991. The Python community has grown significantly since then and there are many people who have contributed to its development, including Guido van Rossum, Ian Grant, Mark Adler, John Lam, and many others.
User: Explain artificial intelligence.
🤖 Chatbot: Artificial intelligence (AI) refers to the ability of machines or computer systems to perform tasks that typically require human intelligence, such as problem-solving, decision-making, and language understanding. AI can be applied to various fields, including business, finance, healthcare, education, and transportation. Some common types of AI include machine learning, natural language processing (NLP), robotics, and computer vision.
Some examples of practical applications of AI include self-driving 